# 01 — Baseline models (single split)

This notebook reproduces **Table 3** of the manuscript: the 13 baseline
models on the single train/validation/test split of each dataset. The same
estimators are also run on the no-protected COMPAS variant (7 inputs instead
of 13), so the single-split benchmark covers both feature sets.

All estimators are instantiated from `utils.baselines`, which is the single
source of truth for their hyperparameters. The values were chosen manually
per dataset by comparing validation accuracy; no automated grid search was
run. Freezing them in one module guarantees that this notebook, the
cross-validation notebook and the manuscript table cannot drift apart.

**Protocol.** The scaler is fitted on the training partition only. Baselines
have fixed hyperparameters, so they are fitted on the training partition and
scored on the test partition. Nothing is tuned on test data.


In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
root = next((p for p in [cwd, *cwd.parents] if (p / 'utils' / 'experiments.py').exists()), cwd)
sys.path.insert(0, str(root))

from utils.baselines import baseline_specs
from utils.experiments import DATASETS, RESULTS_DIR
from scripts.run_baselines import run_dataset, single_split

pd.set_option('display.width', 200)


## 1. Which models are used, and with which hyperparameters

Every entry is `(name, family, factory)`. Print the registry so a reader can
verify a reported number without reading the notebook code.


In [2]:
for key in DATASETS:
    print(f'\n{key} ({len(baseline_specs(key))} baselines)')
    for name, family, build in baseline_specs(key):
        print(f'  {name:24s} [{family}]')



compas (13 baselines)
  Logistic Regression      [Interpretable]
  Decision Tree            [Interpretable]
  EBM (InterpretML)        [Interpretable]
  Naive Bayes              [Interpretable]
  XGBoost                  [SOTA Ensemble]
  LightGBM                 [SOTA Ensemble]
  CatBoost                 [SOTA Ensemble]
  Random Forest            [SOTA Ensemble]
  Gradient Boosting        [SOTA Ensemble]
  AdaBoost                 [SOTA Ensemble]
  SVM (RBF)                [Geometric & DL]
  KNN                      [Geometric & DL]
  MLP (sklearn)            [Geometric & DL]

compas_noprot (13 baselines)
  Logistic Regression      [Interpretable]
  Decision Tree            [Interpretable]
  EBM (InterpretML)        [Interpretable]
  Naive Bayes              [Interpretable]
  XGBoost                  [SOTA Ensemble]
  LightGBM                 [SOTA Ensemble]
  CatBoost                 [SOTA Ensemble]
  Random Forest            [SOTA Ensemble]
  Gradient Boosting        [SOTA Ensemble

## 2. Single-split partitions

Recomputed from the same loaders the manuscript uses.


In [3]:
for key in DATASETS:
    Xtr, ytr, Xva, yva, Xte, yte = single_split(key)
    print(f'{key:10s} train={Xtr.shape[0]:6d}  val={Xva.shape[0]:5d}  test={Xte.shape[0]:6d}  features={Xtr.shape[1]}')


compas     train=  5193  val=  578  test=  1443  features=13
compas_noprot train=  5193  val=  578  test=  1443  features=7
[Data] Loading German Credit Risk...
-> German Credit ready: train shape: (800, 48), Test Shape: (100, 48), feature count: 48, Val Shape: (100, 48)
german     train=   800  val=  100  test=   100  features=48
[Data] Loading UCI Adult Census Income...


-> UCI Adult ready: train shape: (32559, 97), Test Shape: (9045, 97), feature count: 97, Val Shape: (3618, 97)
adult      train= 32559  val= 3618  test=  9045  features=97
synthetic  train= 14400  val= 1600  test=  4000  features=30


## 3. Run (or load) the benchmark

Set `FORCE = True` to refit every model. With `FORCE = False` the cached
`results/baselines_<key>.csv` is reused, which is what the manuscript uses.


In [4]:
FORCE = False

frames = []
for key in DATASETS:
    df = run_dataset(key, force=FORCE)
    frames.append(df.assign(Dataset=DATASETS[key].display_name))

bench = pd.concat(frames, ignore_index=True)
display_cols = ['Dataset', 'Category', 'Model', 'Accuracy', 'F1_Macro', 'ROC_AUC']
bench[display_cols].assign(
    Accuracy=lambda d: (d.Accuracy * 100).round(2),
    F1_Macro=lambda d: (d.F1_Macro * 100).round(2),
    ROC_AUC=lambda d: (d.ROC_AUC * 100).round(2),
).sort_values(['Dataset', 'Accuracy'], ascending=[True, False]).reset_index(drop=True)


[compas] found, skipped (use --force to refresh)
[compas_noprot] found, skipped (use --force to refresh)
[german] found, skipped (use --force to refresh)
[adult] found, skipped (use --force to refresh)
[synthetic] found, skipped (use --force to refresh)


,Dataset,Category,Model,Accuracy,F1_Macro,ROC_AUC
0,COMPAS,Interpretable,Logistic Regression,68.75,67.13,73.22
1,COMPAS,SOTA Ensemble,CatBoost,68.40,67.36,73.40
2,COMPAS,Interpretable,EBM (InterpretML),67.98,67.17,73.10
3,COMPAS,Geometric & DL,MLP (sklearn),67.57,66.90,73.14
4,COMPAS,SOTA Ensemble,XGBoost,67.50,66.67,72.47
...,...,...,...,...,...,...
60,UCI Adult,Geometric & DL,MLP (sklearn),84.75,78.35,90.18
61,UCI Adult,Interpretable,Decision Tree,84.46,76.81,88.16
62,UCI Adult,Geometric & DL,SVM (RBF),84.16,77.43,88.37
63,UCI Adult,Geometric & DL,KNN,82.21,74.74,86.07


## 4. Where the numbers come from

`results/benchmark_comparison.csv` is what `notebooks/07_tables.ipynb` reads
when it writes Table 3. `results/predictions/baselines_<key>_predictions.npz`
holds the per-example predictions used by the McNemar tests.


In [5]:
print(sorted(p.name for p in RESULTS_DIR.glob('baselines_*.csv')))
print(sorted(p.name for p in (RESULTS_DIR / 'predictions').glob('*.npz')))


['baselines_adult.csv', 'baselines_compas.csv', 'baselines_compas_noprot.csv', 'baselines_german.csv', 'baselines_synthetic.csv']
['baselines_adult_predictions.npz', 'baselines_compas_noprot_predictions.npz', 'baselines_compas_predictions.npz', 'baselines_german_predictions.npz', 'baselines_synthetic_predictions.npz']
